# Clase 133 — Segment Anything (SAM / SAM2)

SAM pesa ~2.4 GB (ViT-H). Fallback con SLIC superpixels de skimage que ilustra el concepto de máscara por prompt sobre imagen sintética 256x256.

In [ ]:
USE_SAM = False
try:
    from segment_anything import sam_model_registry, SamPredictor
    USE_SAM = True
    print('segment_anything disponible')
except Exception as e:
    print('SAM no disponible. Fallback SLIC. Motivo:', type(e).__name__)

import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import slic, mark_boundaries
np.random.seed(42)

## 1. Imagen sintética con 3 'objetos'

In [ ]:
img = np.zeros((256, 256, 3), dtype=np.float32)
# fondo gradiente
xx, yy = np.meshgrid(np.linspace(0, 1, 256), np.linspace(0, 1, 256))
img[..., 0] = 0.2 + 0.1*xx; img[..., 1] = 0.3 + 0.1*yy; img[..., 2] = 0.4
# círculo rojo
cx, cy, r = 80, 80, 30
mask_c = (xx*256 - cx)**2 + (yy*256 - cy)**2 < r**2
img[mask_c] = [0.9, 0.1, 0.1]
# rectángulo verde
img[150:200, 60:130] = [0.1, 0.8, 0.2]
# círculo azul
cx2, cy2, r2 = 190, 180, 35
mask_c2 = (xx*256 - cx2)**2 + (yy*256 - cy2)**2 < r2**2
img[mask_c2] = [0.1, 0.2, 0.9]
img = np.clip(img, 0, 1)
plt.imshow(img); plt.title('sintética 256x256'); plt.axis('off'); plt.show()

## 2. Fallback: SLIC superpixels + selección por point prompt

In [ ]:
segments = slic(img, n_segments=30, compactness=15, start_label=0)
print(f'superpixels: {segments.max()+1}')

# Simulamos point prompt en (80, 80) — el centro del círculo rojo
point_xy = (80, 80)
seg_id = segments[point_xy[1], point_xy[0]]
mask = (segments == seg_id)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(mark_boundaries(img, segments)); axes[0].set_title('SLIC'); axes[0].axis('off')
axes[1].imshow(img); axes[1].plot(*point_xy, 'y*', markersize=20); axes[1].set_title('point prompt'); axes[1].axis('off')
axes[2].imshow(img); axes[2].imshow(mask, alpha=0.5, cmap='Reds'); axes[2].set_title(f'máscara superpixel #{seg_id}'); axes[2].axis('off')
plt.show()

## 3. API conceptual SAM

```python
from segment_anything import sam_model_registry, SamPredictor
sam = sam_model_registry['vit_h'](checkpoint='sam_vit_h_4b8939.pth')
predictor = SamPredictor(sam)
predictor.set_image(img_uint8)              # corre el ViT encoder 1 vez
masks, scores, logits = predictor.predict(
    point_coords=np.array([[80, 80]]),       # x, y
    point_labels=np.array([1]),               # 1=foreground, 0=background
    multimask_output=True,                    # 3 hipótesis (whole/part/subpart)
)
best = masks[scores.argmax()]
```

También acepta `box=[x0,y0,x1,y1]` o ambos.

## 4. Múltiples prompts simulados

In [ ]:
prompts = [(80, 80, 'círculo rojo'), (95, 175, 'rectángulo verde'), (190, 180, 'círculo azul')]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (x, y, name) in zip(axes, prompts):
    seg_id = segments[y, x]
    mask = segments == seg_id
    ax.imshow(img); ax.imshow(mask, alpha=0.5, cmap='spring')
    ax.plot(x, y, 'y*', markersize=15)
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. SAM2 — diferencias

SAM2 (Meta, 2024) extiende a video:
- **Memory bank**: propaga máscaras frame-a-frame con un módulo Memory Attention que recuerda objetos pasados.
- **Streaming**: procesa videos en tiempo real (~44 FPS en A100).
- **Image también**: 6x más rápido que SAM v1 en imágenes; mismo prompting (point, box, mask).
- **Promptable Visual Segmentation (PVS)**: un click en frame=0 propaga la máscara a todo el video.

```python
from sam2.sam2_video_predictor import SAM2VideoPredictor
predictor = SAM2VideoPredictor.from_pretrained('facebook/sam2-hiera-large')
state = predictor.init_state(video_path='clip.mp4')
predictor.add_new_points(state, frame_idx=0, obj_id=1, points=[[x,y]], labels=[1])
for frame_idx, obj_ids, masks in predictor.propagate_in_video(state):
    ...
```

## Conclusiones

- SAM = foundation model para segmentación; cualquier prompt (point/box/mask) → máscara.
- Zero-shot: no requiere fine-tuning para clases nuevas.
- SAM2 generaliza a video con propagación temporal vía memory bank.
- Pipeline real (no fallback): `set_image()` 1x → muchos `predict()` baratos.
- En CPU corre pero lento (~30s/imagen ViT-H); GPU recomendada.